In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(FNN)
})

In [ ]:
cds_combined = readRDS("../data/cds_combined_VCS_JAX_mito_removed_all_markers_E15-15.5.RDS")

# Label transfer with co-embedded data

In [ ]:
pca_coordinates = as.data.frame(reducedDims(cds_combined)[['Aligned']])

In [ ]:
pca_coordinates$Cell = rownames(pca_coordinates)

monocle_pca_embeddings =
  as.data.frame(colData(cds_combined)) %>%
            left_join(., pca_coordinates, by = "Cell")

JAX_pca_umap = 
  monocle_pca_embeddings %>%
  filter(Data == "JAX")

VCS_pca_umap = 
  monocle_pca_embeddings %>%
  filter(Data != "JAX") 

In [ ]:
VCS_pca_umap %>%
    select(PC1:PC30) %>%
    head()

In [ ]:
# Get nearest neighbors using the KNN package
nn = 
  get.knnx(data = JAX_pca_umap %>%
             dplyr::select(PC1:PC30) %>%
             as.matrix(),
           query = VCS_pca_umap %>%
             dplyr::select(PC1:PC30) %>%
             as.matrix(),
           k = 10)

In [ ]:
head(nn$nn.index)

In [ ]:
head(nn$nn.dist)

In [ ]:
## Use nearest neighbor indices to get top 10 labels
nn_cell_type = 
  sapply(X = seq(1,dim(nn$nn.index)[2]),
         FUN = function(X){
           JAX_pca_umap$Main_cell_type[nn$nn.index[,X]]
         }
  ) %>%
  as.data.frame()

## Use nearest neighbor indices to get top 10 labels
nn_trajectory = 
  sapply(X = seq(1,dim(nn$nn.index)[2]),
         FUN = function(X){
           JAX_pca_umap$Main_trajectory[nn$nn.index[,X]]
         }
  ) %>%
  as.data.frame()


In [ ]:
rownames(nn_cell_type) = VCS_pca_umap$Cell
rownames(nn_trajectory) = VCS_pca_umap$Cell

In [ ]:
head(nn_cell_type)

In [ ]:
# Collect the results -----------------------------------------------------
# Take the majority celltype annotation for each cell across all the iterations 
nn_cell_type_majority = 
  sapply(X = seq(1,dim(nn_cell_type)[1]),
         FUN = function(X){
           tabulated_labels = 
             nn_cell_type[X,] %>%
             unlist() %>%
             table() %>%
             sort(decreasing = T)
           
           tabulated_labels[1] %>%
             names()
         })

# Collect the results -----------------------------------------------------
# Take the majority celltype annotation for each cell across all the iterations 
nn_trajectory_majority = 
  sapply(X = seq(1,dim(nn_trajectory)[1]),
         FUN = function(X){
           tabulated_labels = 
             nn_trajectory[X,] %>%
             unlist() %>%
             table() %>%
             sort(decreasing = T)
           
           tabulated_labels[1] %>%
             names()
         })

In [ ]:
# Calculate the proportion that of nearest neighbors in the majority label across all iterations 
majority_fraction_cell_type =
  sapply(X = seq(1,dim(nn_cell_type)[1]), 
                  FUN = function(Y){
                    sum(nn_cell_type[Y,] == nn_cell_type_majority[Y])/10
                  
                    })

# Calculate the proportion that of nearest neighbors in the majority label across all iterations 
majority_fraction_trajectory =
  sapply(X = seq(1,dim(nn_trajectory)[1]), 
                  FUN = function(Y){
                    sum(nn_trajectory[Y,] == nn_trajectory_majority[Y])/10
                  
                    })
estimates =
  data.frame(Cell = VCS_pca_umap$Cell,
           majority_fraction_cell_type = majority_fraction_cell_type,
           nn_majority_cell_type = nn_cell_type_majority, 
           majority_fraction_trajectory = majority_fraction_trajectory,
           nn_majority_trajectory = nn_trajectory_majority)

In [ ]:
write.table(x = estimates,
            file = "alignment_coldata_pca_JAX.tsv",
            sep = "\t",
            quote = F,
            row.names = F,
            col.names = T)

# UMAP coordinates

In [ ]:
pca_coordinates = as.data.frame(reducedDims(cds_combined)[['UMAP']])

In [ ]:
pca_coordinates$Cell = rownames(pca_coordinates)

monocle_pca_embeddings =
  as.data.frame(colData(cds_combined)) %>%
            left_join(., pca_coordinates, by = "Cell")

JAX_pca_umap = 
  monocle_pca_embeddings %>%
  filter(Data == "JAX")

VCS_pca_umap = 
  monocle_pca_embeddings %>%
  filter(Data != "JAX") 

In [ ]:
VCS_pca_umap %>%
    select(V1:V3) %>%
    head()

In [ ]:
# Get nearest neighbors using the KNN package
nn = 
  get.knnx(data = JAX_pca_umap %>%
             dplyr::select(V1:V3) %>%
             as.matrix(),
           query = VCS_pca_umap %>%
             dplyr::select(V1:V3) %>%
             as.matrix(),
           k = 20)

In [ ]:
head(nn$nn.index)

In [ ]:
head(nn$nn.dist)

In [ ]:
## Use nearest neighbor indices to get top 10 labels
nn_cell_type = 
  sapply(X = seq(1,dim(nn$nn.index)[2]),
         FUN = function(X){
           JAX_pca_umap$Main_cell_type[nn$nn.index[,X]]
         }
  ) %>%
  as.data.frame()

## Use nearest neighbor indices to get top 10 labels
nn_trajectory = 
  sapply(X = seq(1,dim(nn$nn.index)[2]),
         FUN = function(X){
           JAX_pca_umap$Main_trajectory[nn$nn.index[,X]]
         }
  ) %>%
  as.data.frame()


In [ ]:
rownames(nn_cell_type) = VCS_pca_umap$Cell
rownames(nn_trajectory) = VCS_pca_umap$Cell

In [ ]:
head(nn_cell_type)

In [ ]:
# Collect the results -----------------------------------------------------
# Take the majority celltype annotation for each cell across all the iterations 
nn_cell_type_majority = 
  sapply(X = seq(1,dim(nn_cell_type)[1]),
         FUN = function(X){
           tabulated_labels = 
             nn_cell_type[X,] %>%
             unlist() %>%
             table() %>%
             sort(decreasing = T)
           
           tabulated_labels[1] %>%
             names()
         })

# Collect the results -----------------------------------------------------
# Take the majority celltype annotation for each cell across all the iterations 
nn_trajectory_majority = 
  sapply(X = seq(1,dim(nn_trajectory)[1]),
         FUN = function(X){
           tabulated_labels = 
             nn_trajectory[X,] %>%
             unlist() %>%
             table() %>%
             sort(decreasing = T)
           
           tabulated_labels[1] %>%
             names()
         })

In [ ]:
# Calculate the proportion that of nearest neighbors in the majority label across all iterations 
majority_fraction_cell_type =
  sapply(X = seq(1,dim(nn_cell_type)[1]), 
                  FUN = function(Y){
                    sum(nn_cell_type[Y,] == nn_cell_type_majority[Y])/20
                  
                    })

# Calculate the proportion that of nearest neighbors in the majority label across all iterations 
majority_fraction_trajectory =
  sapply(X = seq(1,dim(nn_trajectory)[1]), 
                  FUN = function(Y){
                    sum(nn_trajectory[Y,] == nn_trajectory_majority[Y])/20
                  
                    })
estimates =
  data.frame(Cell = VCS_pca_umap$Cell,
           majority_fraction_cell_type = majority_fraction_cell_type,
           nn_majority_cell_type = nn_cell_type_majority, 
           majority_fraction_trajectory = majority_fraction_trajectory,
           nn_majority_trajectory = nn_trajectory_majority)

In [ ]:
write.table(x = estimates,
            file = "alignment_coldata_umap_JAX.tsv",
            sep = "\t",
            quote = F,
            row.names = F,
            col.names = T)

## Assign labels

In [ ]:
cds_combined = readRDS("../data/cds_combined_VCS_JAX_mito_removed_all_markers_E15-15.5.RDS")

# Read in the nearest neighbor assignments
alignment_coldata_pca = 
  read.table(file = "alignment_coldata_pca_JAX.tsv",
             sep = "\t",
             header = T) 

alignment_coldata_umap = 
  read.table(file = "alignment_coldata_umap_JAX.tsv",
             sep = "\t",
             header = T) 

In [ ]:
cds_combined$nn_majority_cell_type_pca = 
  colData(cds_combined) %>%
  as.data.frame() %>%
  left_join(alignment_coldata_pca %>%
              dplyr::select(Cell,
                    nn_majority_cell_type),
            by = "Cell") %>%
  pull(nn_majority_cell_type) %>%
  as.character()

cds_combined$nn_majority_trajectory_pca = 
  colData(cds_combined) %>%
  as.data.frame() %>%
  left_join(alignment_coldata_pca %>%
              dplyr::select(Cell,
                    nn_majority_trajectory),
            by = "Cell") %>%
  pull(nn_majority_trajectory) %>%
  as.character()


cds_combined$nn_majority_cell_type_umap = 
  colData(cds_combined) %>%
  as.data.frame() %>%
  left_join(alignment_coldata_umap %>%
              dplyr::select(Cell,
                    nn_majority_cell_type),
            by = "Cell") %>%
  pull(nn_majority_cell_type) %>%
  as.character()

cds_combined$nn_majority_trajectory_umap = 
  colData(cds_combined) %>%
  as.data.frame() %>%
  left_join(alignment_coldata_umap %>%
              dplyr::select(Cell,
                    nn_majority_trajectory),
            by = "Cell") %>%
  pull(nn_majority_trajectory) %>%
  as.character()